# BÀI TẬP LỚN MÔN KHAI PHÁ DỮ LIỆU (PTIT)
## ĐỀ TÀI SỐ 8: PHÁT HIỆN TIN NHẮN RÁC (SMS SPAM DETECTION)
---
**Tập dữ liệu:** UCI Machine Learning Repository - SMS Spam Collection (5.574 tin nhắn thô)

**Bám sát bài giảng môn học:**
- `Data.pptx`: Làm sạch dữ liệu (Slide 61-66), Thống kê mô tả (Slide 12-38), Chọn thuộc tính $\chi^2$ (Slide 20, 101), Chuẩn hóa Min-Max (Slide 74)
- `Classification.pptx`: Naive Bayes & Laplace (Slide 22, 29), Decision Tree & Entropy (Slide 8-18), Linear SVM, Đánh giá Holdout & 10-Fold CV & ROC-AUC (Slide 52-62), Quá khớp Overfitting (Slide 59)

> *Notebook này có thể chạy hoàn toàn miễn phí trên Google Colab hoặc Jupyter Notebook / VS Code cá nhân.*

### BƯỚC 1: Cài đặt thư viện & Tải tập dữ liệu
Tự động tải bộ dữ liệu `SMSSpamCollection` từ kho lưu trữ của Đại học California (UCI).

In [ ]:
# 1. Cài đặt các thư viện cần thiết (nếu chạy trên Colab đã có sẵn hầu hết)
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk

import os
import io
import zipfile
import urllib.request
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve, roc_auc_score
)

# Tải dữ liệu từ UCI nếu chưa có
DATA_PATH = 'SMSSpamCollection'
DATA_URL = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'

if not os.path.exists(DATA_PATH):
    print('[*] Đang tải dữ liệu từ UCI Repository...')
    raw = urllib.request.urlopen(DATA_URL, timeout=30).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        with open(DATA_PATH, 'wb') as f:
            f.write(z.read('SMSSpamCollection'))
    print('[+] Đã tải và giải nén thành công!')
else:
    print('[+] Dữ liệu đã sẵn sàng.')

# Đọc dữ liệu
df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['label', 'message'], quoting=3, encoding='utf-8')
print(f'Kích thước ban đầu: {df.shape[0]} dòng, {df.shape[1]} cột.')
df.head()

### BƯỚC 2: Làm sạch dữ liệu (Data Cleaning)
*Tham chiếu: `Data.pptx` (Slide 61-66)*
- Xử lý giá trị khuyết thiếu (Missing values)
- Xử lý các bản ghi trùng lặp (Duplicate records)
- Mã hóa nhãn nhị phân bất đối xứng: `ham` = 0 (tin thường), `spam` = 1 (tin rác) *(Slide 9)*

In [ ]:
# Kiểm tra dữ liệu thiếu và trùng lặp
n_missing = df.isna().any(axis=1).sum()
n_dup = df.duplicated().sum()
print(f'Số dòng thiếu: {n_missing} | Số dòng trùng lặp: {n_dup}')

# Loại bỏ dữ liệu rác
df = df.dropna().drop_duplicates().reset_index(drop=True)
df['target'] = (df['label'] == 'spam').astype(int)

print(f'-> Số tin nhắn sau khi làm sạch: {len(df)}')
print(df['label'].value_counts())
print(f"Tỷ lệ mất cân bằng: {df['label'].value_counts(normalize=True).round(3).to_dict()}")

### BƯỚC 3: Trích xuất đặc trưng Metadata & Thống kê mô tả (EDA)
*Tham chiếu: `Data.pptx` (Slide 12-38: Statics of Data & Slide 73: Feature Construction)*
- Xây dựng các thuộc tính: độ dài ký tự, số từ, số chữ số, số chữ in hoa, có chứa link URL, có chứa ký tự tiền tệ (`$`, `£`, `€`).
- Phân tích tóm tắt 5 số (Five-number summary: Min, Q1, Median, Q3, Max) và điểm dị biệt ngoại lai ($1.5 \times \text{IQR}$).
- Đo lường hệ số tương quan Pearson với nhãn Spam *(Slide 29)*.

In [ ]:
# Xây dựng các đặc trưng metadata (Feature Construction)
df['num_chars']   = df['message'].str.len()
df['num_words']   = df['message'].str.split().str.len()
df['num_upper']   = df['message'].str.count(r'[A-Z]')
df['num_digits']  = df['message'].str.count(r'\d')
df['num_special'] = df['message'].str.count(r'[^A-Za-z0-9\s]')
df['has_url']     = df['message'].str.contains(r'http|www\.', case=False).astype(int)
df['has_money']   = df['message'].str.contains(r'[£$€]').astype(int)

META_COLS = ['num_chars', 'num_words', 'num_upper', 'num_digits', 'num_special', 'has_url', 'has_money']

# Bảng thống kê mô tả 5 số của độ dài ký tự
g = df.groupby('label')['num_chars']
stats = pd.DataFrame({
    'Mean': g.mean(), 'Median': g.median(),
    'Std': g.std(), 'Min': g.min(),
    'Q1': g.quantile(0.25), 'Q3': g.quantile(0.75), 'Max': g.max()
}).round(1)
print('BẢNG THỐNG KÊ 5 SỐ VÀ PHÂN TÁN (Slide 14-33):')
display(stats)

# Hệ số tương quan Pearson với nhãn spam
corr = df[META_COLS + ['target']].corr()['target'].drop('target').sort_values(ascending=False)
print('\nHỆ SỐ TƯƠNG QUAN PEARSON VỚI NHÃN SPAM (Slide 29):')
print(corr.round(3))

# Trực quan hóa (Slide 32-35)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
pal = {'ham': '#4C72B0', 'spam': '#DD5555'}

sns.countplot(x='label', hue='label', data=df, palette=pal, legend=False, ax=axes[0])
axes[0].set_title('Phân bố nhãn (Mất cân bằng lớp)')

sns.histplot(data=df, x='num_chars', hue='label', bins=50, palette=pal, element='step', ax=axes[1])
axes[1].set_xlim(0, 400)
axes[1].set_title('Histogram độ dài tin nhắn (Slide 34)')

sns.boxplot(data=df, x='label', y='num_chars', hue='label', palette=pal, legend=False, ax=axes[2])
axes[2].set_ylim(0, 400)
axes[2].set_title('Boxplot phát hiện dị biệt (Slide 33)')

plt.tight_layout()
plt.show()

### BƯỚC 4: Tiền xử lý văn bản (Text Preprocessing)
*Bám sát yêu cầu đề bài của Thầy: (tokenization, stop-word removal, lemmatization/stemming)*
- Đưa về chữ thường (`lower`)
- Bỏ các ký tự đặc biệt, dấu câu, số (do đã đếm ở Metadata)
- Bỏ từ dừng tiếng Anh (`stop_words`)
- Đưa từ về dạng gốc bằng giải thuật **Porter Stemmer (1980)** để giảm chiều dữ liệu *(Data.pptx Slide 96: Dimensionality Reduction)*

In [ ]:
stemmer = PorterStemmer()

def preprocess_text(text):
    # 1. Chuyển chữ thường và bỏ ký tự lạ
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 2. Tokenization & loại bỏ stop words
    tokens = text.split()
    tokens = [w for w in tokens if w not in ENGLISH_STOP_WORDS and len(w) > 1]
    # 3. Stemming
    return ' '.join(stemmer.stem(w) for w in tokens)

df['clean'] = df['message'].apply(preprocess_text)

# Xem thử 2 mẫu trước và sau khi làm sạch
for i, r in df[df.target == 1].head(2).iterrows():
    print(f'Gốc : {r.message[:90]}...')
    print(f'Sạch: {r.clean[:90]}...\n')

### BƯỚC 5: Xây dựng Pipeline trích xuất đặc trưng
- **Văn bản (Bag-of-Words):** Dùng `CountVectorizer` để đếm tần số xuất hiện của từ *(Data.pptx Slide 2, 54)*.
- **Chọn lọc đặc trưng:** Dùng kiểm định **Chi-Square ($\chi^2$)** để lọc ra 1.000 từ quan trọng nhất *(Data.pptx Slide 20, 101)*.
- **Metadata:** Chuẩn hóa bằng **Min-Max Normalization** đưa về khoảng $[0, 1]$ *(Data.pptx Slide 74)*.

In [ ]:
K_WORDS = 1000

# Bộ xử lý kết hợp cả văn bản lẫn metadata
feature_preprocessor = ColumnTransformer([
    ('bow', Pipeline([
        ('count', CountVectorizer(min_df=2)),
        ('chi2', SelectKBest(chi2, k=K_WORDS))
    ]), 'clean'),
    ('meta', MinMaxScaler(), META_COLS)
])

print(f'[+] Pipeline đã cấu hình: Giữ lại top {K_WORDS} từ có Chi-Square cao nhất + 7 thuộc tính metadata.')

### BƯỚC 6: Huấn luyện & Đánh giá 3 mô hình
*Tham chiếu: `Classification.pptx` (Slide 8-50 & 52-62)*
1. **Multinomial Naive Bayes (kèm làm mịn Laplace $\alpha = 1$)** *(Slide 22, 29)*
2. **Linear SVM (Tìm siêu phẳng khoảng cách lề lớn nhất)** *(Chương 9 sách Han)*
3. **Decision Tree (Dựa trên Entropy / Information Gain, giới hạn độ sâu `max_depth=15`)** *(Slide 8-18)*

Phương pháp phân chia dữ liệu: **Holdout phân tầng (Stratified Holdout: 2/3 Train - 1/3 Test)** *(Slide 60)*.

In [ ]:
# Chia dữ liệu theo Holdout phân tầng (2/3 Train, 1/3 Test)
X = df[['clean'] + META_COLS]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1/3, stratify=y, random_state=42
)
print(f'Tập Train: {len(X_train)} mẫu | Tập Test: {len(X_test)} mẫu.')

models = {
    'Naive Bayes': MultinomialNB(alpha=1.0),
    'Linear SVM': LinearSVC(C=1.0, random_state=42),
    'Decision Tree': DecisionTreeClassifier(criterion='entropy', max_depth=15, random_state=42)
}

holdout_results = {}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plt_roc = plt.figure(figsize=(7, 5)).gca()

for i, (name, clf) in enumerate(models.items()):
    # Tạo pipeline hoàn chỉnh
    pipe = Pipeline([('features', feature_preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    # Lấy điểm số phục vụ vẽ ROC
    scores = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe.named_steps['clf'], 'predict_proba') else pipe.decision_function(X_test)
    
    # Tính các chỉ số theo Slide 53-62
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    holdout_results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Error rate': (fp + fn) / len(y_test),
        'Sensitivity (Recall)': recall_score(y_test, y_pred),
        'Specificity': tn / (tn + fp),
        'Precision': precision_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, scores)
    }
    
    # Vẽ ma trận nhầm lẫn (Slide 53)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[i],
                xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
    axes[i].set_title(f'{name}\nFP={cm[0, 1]}  FN={cm[1, 0]}')
    axes[i].set_xlabel('Predicted class')
    axes[i].set_ylabel('Actual class')
    
    # Vẽ đường cong ROC (Slide 62)
    fpr, tpr, _ = roc_curve(y_test, scores)
    plt_roc.plot(fpr, tpr, label=f"{name} (AUC={holdout_results[name]['AUC']:.3f})")

plt_roc.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt_roc.set_xlabel('False positive rate (FP / N)')
plt_roc.set_ylabel('True positive rate (TP / P)')
plt_roc.set_title('Đường cong ROC của 3 mô hình (Slide 62)')
plt_roc.legend()

fig.tight_layout()
plt.show()

print('BẢNG KẾT QUẢ ĐÁNH GIÁ HOLDOUT:')
display(pd.DataFrame(holdout_results).T.round(4))

### BƯỚC 7: Thẩm định chéo Stratified 10-Fold Cross-Validation
*Tham chiếu: `Classification.pptx` (Slide 61)*
Chia 10 phần độc lập để đánh giá độ ổn định của các mô hình (Báo cáo dạng: Trung bình ± Độ lệch chuẩn).

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_rows = {}

for name, clf in models.items():
    pipe = Pipeline([('features', feature_preprocessor), ('clf', clf)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'])
    cv_rows[name] = {
        'Accuracy': f"{scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}",
        'Precision': f"{scores['test_precision'].mean():.4f} ± {scores['test_precision'].std():.4f}",
        'Recall': f"{scores['test_recall'].mean():.4f} ± {scores['test_recall'].std():.4f}",
        'F1-Score': f"{scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}"
    }

print('KẾT QUẢ 10-FOLD CROSS-VALIDATION (Slide 61):')
display(pd.DataFrame(cv_rows).T)

### BƯỚC 8: Hai thí nghiệm thực chứng trực tiếp lý thuyết bài giảng
#### Thí nghiệm 1: Kiểm chứng vai trò của kỹ thuật làm mịn Laplace
*Tham chiếu: `Classification.pptx` (Slide 29: Avoiding the Zero-Probability Problem)*

In [ ]:
print('--- THÍ NGHIỆM 1: LAPLACIAN CORRECTION ---')
for alpha, desc in [(1e-10, 'Không làm mịn (alpha ≈ 0)'), (1.0, 'Có làm mịn Laplace (alpha = 1.0)')]:
    pipe_nb = Pipeline([('features', feature_preprocessor), ('clf', MultinomialNB(alpha=alpha))])
    pipe_nb.fit(X_train, y_train)
    y_pred = pipe_nb.predict(X_test)
    print(f"{desc:<35} -> Precision: {precision_score(y_test, y_pred):.4f} | Recall: {recall_score(y_test, y_pred):.4f} | F1: {f1_score(y_test, y_pred):.4f}")

#### Thí nghiệm 2: Kiểm chứng hiện tượng quá khớp (Overfitting) của Cây quyết định
*Tham chiếu: `Classification.pptx` (Slide 13 & Slide 59: Training Error vs Testing Error)*

In [ ]:
depths = [1, 2, 3, 5, 8, 12, 15, 20, 30, 50, None]
tr_err, te_err = [], []

for d in depths:
    dt = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=42)
    pipe_dt = Pipeline([('features', feature_preprocessor), ('clf', dt)])
    pipe_dt.fit(X_train, y_train)
    tr_err.append(1 - pipe_dt.score(X_train, y_train))
    te_err.append(1 - pipe_dt.score(X_test, y_test))

labels = [str(d) if d else '∞' for d in depths]

plt.figure(figsize=(8, 4.5))
plt.plot(labels, tr_err, 'o-', color='red', label='Training error')
plt.plot(labels, te_err, 's-', color='green', label='Testing error')
plt.xlabel('Độ sâu tối đa (max_depth)')
plt.ylabel('Tỷ lệ lỗi (Error Rate)')
plt.title('Minh họa hiện tượng Quá khớp (Overfitting) - Slide 59')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print('Nhận xét: Khi max_depth >= 20, Training Error giảm về 0 (học vẹt), nhưng Testing Error tăng vọt trở lại!')

### BƯỚC 9: Thử nghiệm Demo dự đoán tin nhắn thực tế
Dùng mô hình tốt nhất (Linear SVM - Precision 96.6%) để dự đoán một vài tin nhắn mẫu.

In [ ]:
# Huấn luyện mô hình tốt nhất trên toàn bộ dữ liệu
best_model = Pipeline([('features', feature_preprocessor), ('clf', LinearSVC(C=1.0, random_state=42))])
best_model.fit(X, y)

test_messages = [
    'Hey, are we still meeting for lunch today?',
    'CONGRATULATIONS! You won a $1000 Walmart gift card. Call 09061701461 now to claim your prize!',
    'Can you send me the slides from yesterday lecture?',
    'URGENT! Your mobile account has been suspended. Please text YES to 88088 to verify.'
]

df_demo = pd.DataFrame({'message': test_messages})
df_demo['num_chars']   = df_demo['message'].str.len()
df_demo['num_words']   = df_demo['message'].str.split().str.len()
df_demo['num_upper']   = df_demo['message'].str.count(r'[A-Z]')
df_demo['num_digits']  = df_demo['message'].str.count(r'\d')
df_demo['num_special'] = df_demo['message'].str.count(r'[^A-Za-z0-9\s]')
df_demo['has_url']     = df_demo['message'].str.contains(r'http|www\.', case=False).astype(int)
df_demo['has_money']   = df_demo['message'].str.contains(r'[£$€]').astype(int)
df_demo['clean']       = df_demo['message'].apply(preprocess_text)

predictions = best_model.predict(df_demo[['clean'] + META_COLS])

print('KẾT QUẢ DỰ ĐOÁN THỰC TẾ:')
for msg, pred in zip(test_messages, predictions):
    tag = '[ SPAM ]' if pred == 1 else '[  HAM ]'
    print(f"{tag} {msg}")